# 07 — Report Figures

Generates all tables and plots required for the written report.
Each figure function is independent — runs from saved output files
and can be re-run individually after any upstream change.

| Fig | Output | Requires |
|-----|--------|---------|
| 1a | `model_performance_primary.csv` + `fig1_model_performance.png` | `model_comparison_by_threshold.csv` |
| 1b | `fig1b_rmse_heatmap_all_thresholds.png` | same |
| 2  | `fig2_feature_importance_consensus.png` | `shap_feature_importance_comparison.csv` |
| 3  | `fig3_trajectory_{country}.png` × 5 | `poverty_predictions_ssp.csv` |
| 4  | `fig4_regional_trends.png` | `poverty_predictions_ssp.csv` |
| 5  | `fig5_approach_divergence.png` | `approach_comparison.csv` |
| 6  | `fig6_learning_curve.png` | `X_train.csv`, `y_train.csv`, model pkl |
| 7  | `fig7_residual_analysis.png` | `X_test.csv`, `y_test.csv`, model pkl |


## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from report_figures import (
    generate_all_figures,
    fig1_model_performance_table,
    fig2_feature_importance_consensus,
    fig3_prediction_trajectories,
    fig4_regional_trends,
    fig5_approach_divergence,
    fig6_learning_curve,
    fig7_residual_analysis,
    REPORT_DIR,
    MODEL_DISPLAY,
    THRESHOLD_DISPLAY,
)
from config import DATA_FINAL_DIR, OUTPUTS_DIR

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Report output dir: {REPORT_DIR}")

# Check what upstream outputs are available
to_check = [
    OUTPUTS_DIR / "model_comparison_by_threshold.csv",
    OUTPUTS_DIR / "shap_feature_importance_comparison.csv",
    OUTPUTS_DIR / "approach_comparison.csv",
    DATA_FINAL_DIR / "poverty_predictions_ssp.csv",
    DATA_FINAL_DIR / "X_train.csv",
    DATA_FINAL_DIR / "X_test.csv",
]
print("\nUpstream file status:")
for p in to_check:
    exists = p.exists()
    size   = f"{p.stat().st_size:>10,} bytes" if exists else "NOT FOUND"
    print(f"  {'✓' if exists else '✗'}  {p.name:<50}  {size}")

## 1. Model performance table (Fig 1)

In [ ]:
tables = fig1_model_performance_table(OUTPUTS_DIR)

if tables:
    print("Primary table ($3/day):")
    display(tables["primary"])
    print("\nAll thresholds (RMSE):")
    display(tables["wide"])

### 1b. Figure inline

In [ ]:
from IPython.display import Image, display
for fname in ["fig1_model_performance.png", "fig1b_rmse_heatmap_all_thresholds.png"]:
    p = REPORT_DIR / fname
    if p.exists():
        display(Image(str(p)))

## 2. Feature importance consensus (Fig 2)

In [ ]:
imp = fig2_feature_importance_consensus(OUTPUTS_DIR)
if imp is not None:
    model_cols = [m for m in ["xgboost_cpu","xgboost_gpu","lightgbm",
                               "random_forest","gam","ridge","mlp"]
                  if m in imp.columns]
    display(imp[["feature_label"] + model_cols + ["mean_across_models","rank_mean"]].round(4))

In [ ]:
p = REPORT_DIR / "fig2_feature_importance_consensus.png"
if p.exists():
    display(Image(str(p)))

## 3. Country trajectory plots (Fig 3)

In [ ]:
COUNTRIES = ["Nigeria", "India", "Brazil", "Germany", "United States"]
fig3_prediction_trajectories(DATA_FINAL_DIR, countries=COUNTRIES)

for country in COUNTRIES:
    safe = country.replace(" ", "_").replace(".", "")
    p = REPORT_DIR / f"fig3_trajectory_{safe}.png"
    if p.exists():
        print(f"\n{country}")
        display(Image(str(p)))

## 4. Regional trends (Fig 4)

In [ ]:
agg = fig4_regional_trends(DATA_FINAL_DIR)
p = REPORT_DIR / "fig4_regional_trends.png"
if p.exists():
    display(Image(str(p)))

### 4a. Regional summary table

In [ ]:
if agg is not None:
    summary = (
        agg[agg["year"].isin([2030, 2050, 2075, 2100])]
        .pivot_table(index=["region","scenario"], columns="year",
                     values="predicted_poverty")
        .round(2)
    )
    display(summary)

## 5. Approach A vs B divergence (Fig 5)

In [ ]:
fig5_approach_divergence(OUTPUTS_DIR, threshold="$3", country="Nigeria")
p = REPORT_DIR / "fig5_approach_divergence.png"
if p.exists():
    display(Image(str(p)))

### 5a. Divergence for a second country

In [ ]:
fig5_approach_divergence(OUTPUTS_DIR, threshold="$3", country="India")
p = REPORT_DIR / "fig5_approach_divergence.png"
if p.exists():
    display(Image(str(p)))

## 6. Learning curve (Fig 6)

In [ ]:
fig6_learning_curve(DATA_FINAL_DIR, OUTPUTS_DIR)
p = REPORT_DIR / "fig6_learning_curve.png"
if p.exists():
    display(Image(str(p)))
else:
    print("Learning curve not generated — check that model pkl exists.")

## 7. Residual analysis (Fig 7)

In [ ]:
fig7_residual_analysis(DATA_FINAL_DIR, OUTPUTS_DIR)
p = REPORT_DIR / "fig7_residual_analysis.png"
if p.exists():
    display(Image(str(p)))

## 8. Run all figures at once

In [ ]:
# Convenience function — runs everything and prints a file listing
results = generate_all_figures(
    final_dir   = DATA_FINAL_DIR,
    outputs_dir = OUTPUTS_DIR,
    countries   = ["Nigeria", "India", "Brazil", "Germany", "United States"],
)

## 9. Final report file listing

In [ ]:
print("=== Report outputs in outputs/report/ ===")
if REPORT_DIR.exists():
    for p in sorted(REPORT_DIR.rglob("*")):
        print(f"  {p.name:<55}  {p.stat().st_size:>9,} bytes")
else:
    print("Report directory not created yet — run generate_all_figures() first.")

## 10. Figure descriptions for the report

### Fig 1 — Model Performance
Primary comparison table and bar chart: RMSE, MAE, R², MAPE on the $3/day
hold-out test set (2016–2022). The supplementary heatmap extends this to all
4 poverty thresholds. Use to argue which model is selected as the "best" and why.

### Fig 2 — Feature Importance Consensus
Grouped bar chart showing mean |SHAP| per feature across all 7 models.
Features sorted by cross-model mean importance. Use to show that GDP per capita
and Gini consistently dominate regardless of model family — lending robustness
to the findings.

### Fig 3 — Country Trajectories
Country-level poverty trajectories under SSP1/4/5. Nigeria and Ethiopia show
diverging scenarios (SSP1 much lower poverty than SSP4/5). Germany and USA show
very low poverty across all scenarios — confirming sanity of predictions.

### Fig 4 — Regional Trends
Aggregates predictions by World Bank region. Sub-Saharan Africa shows the
highest absolute poverty levels; South Asia shows rapid reduction particularly
under SSP1. Use to discuss geographic concentration of poverty risk.

### Fig 5 — Approach A vs B
Shows tree models producing flat predictions after 2050 (plateauing once features
exit the training distribution), while Ridge/MLP continue to extrapolate.
Increasing cross-model spread is used as a proxy for forecast uncertainty.

### Fig 6 — Learning Curve
Diagnoses the bias-variance tradeoff: does the model benefit from more training
data? A converging train/val gap suggests adequate sample size; a persistent gap
suggests the model could benefit from more data or regularisation.

### Fig 7 — Residual Analysis
Checks whether residuals are well-behaved. The regional boxplot reveals any
systematic geographic bias (e.g. consistently over-predicting poverty in one
region). The Q-Q plot checks normality of errors — important for uncertainty
quantification.
